In [ ]:
!pip install wandb==0.21.1 scikit-learn==1.6.1 torch==2.6.0+cu124

In [ ]:
!wandb login

### DATA INGESTION

In [ ]:
!uv add openpyxl

In [1]:
import pandas as pd
import wandb

# URL for a sample retail sales dataset
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"



from datetime import datetime

run_name = f"ingest-raw-sales-{datetime.now().strftime('%Y%m%d-%H%M')}"

def ingest_raw_data():
    """
    Downloads raw sales data and logs it as a W&B Artifact.
    """
    with wandb.init(project="sales-forecasting", job_type="ingest-data",name=run_name) as run:
        # Download data
        print("Downloading raw data...")
        df = pd.read_excel(DATA_URL)

        # Save to local file
        filename = "raw_sales_data.csv"
        df.to_csv(filename, index=False)

        # Create and log artifact
        raw_data_artifact = wandb.Artifact(
            "raw-sales-data",
            type="raw_dataset",
            description="Raw online retail sales data from UCI."
        )
        raw_data_artifact.add_file(filename)
        run.log_artifact(raw_data_artifact)
        print("Raw data artifact logged.")

ingest_raw_data()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\jagan\_netrc.
wandb: Currently logged in as: jagansairohandevaki (jagansairohandevaki-google) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Raw data artifact logged.


### DATA PREPROCESSING

In [2]:
import pandas as pd
import wandb
from sklearn.preprocessing import StandardScaler

def preprocess_data():
    """
    Consumes the raw data artifact, preprocesses it for time series forecasting,
    and logs a new processed data artifact.
    """
    with wandb.init(project="sales-forecasting", job_type="preprocess-data") as run:
        # Use the raw data artifact
        raw_artifact = run.use_artifact('raw-sales-data:latest')
        raw_data_dir = raw_artifact.download()
        df = pd.read_csv(f"{raw_data_dir}/raw_sales_data.csv")

        # Initial count
        original_row_count = len(df)
        print(f"Total raw rows: {original_row_count}")

        # --- Data Cleaning ---
        missing_customer_rows = df["CustomerID"].isna().sum()
        df = df.dropna(subset=["CustomerID"])

        negative_quantity_rows = (df["Quantity"] <= 0).sum()
        df = df[df["Quantity"] > 0]

        nonpositive_price_rows = (df["UnitPrice"] <= 0).sum()
        df = df[df["UnitPrice"] > 0]

        cleaned_row_count = len(df)

        print(f"Rows dropped due to missing CustomerID: {missing_customer_rows}")
        print(f"Rows dropped due to non-positive Quantity: {negative_quantity_rows}")
        print(f"Rows dropped due to non-positive UnitPrice: {nonpositive_price_rows}")
        print(f"Remaining rows after cleaning: {cleaned_row_count}")

        # --- Time Series Aggregation ---
        df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
        df["Sales"] = df["Quantity"] * df["UnitPrice"]
        # Resample: aggregate total daily sales across all products/customers
        daily_sales = df.set_index("InvoiceDate")["Sales"].resample("D").sum().reset_index()
        daily_sales = daily_sales.rename(columns={"InvoiceDate": "Date", "Sales": "TotalSales"})
        daily_sales = daily_sales.set_index("Date").asfreq("D", fill_value=0).reset_index()

        total_days = len(daily_sales)
        non_zero_days = (daily_sales["TotalSales"] > 0).sum()
        date_start = daily_sales["Date"].min()
        date_end = daily_sales["Date"].max()

        print(f"\nResampled daily sales from {date_start.date()} to {date_end.date()}")
        print(f"Total days after resampling: {total_days}")
        print(f"Days with non-zero sales: {non_zero_days}")

        # Simple train/validation split (e.g., last 60 days for validation)
        train_df = daily_sales[:-60]
        val_df = daily_sales[-60:]

        print(f"\nTrain set rows: {len(train_df)}")
        print(f"Validation set rows: {len(val_df)}")

        # Save processed files
        train_df.to_csv("train.csv", index=False)
        val_df.to_csv("validation.csv", index=False)

        wandb.log({
            "original_row_count": original_row_count,
            "rows_dropped_missing_customer": missing_customer_rows,
            "rows_dropped_negative_quantity": negative_quantity_rows,
            "rows_dropped_nonpositive_price": nonpositive_price_rows,
            "cleaned_row_count": len(df)
        })

        # After resampling
        wandb.log({
            "resample_start_date": date_start.strftime("%Y-%m-%d"),
            "resample_end_date": date_end.strftime("%Y-%m-%d"),
            "resample_total_days": total_days,
            "resample_non_zero_days": non_zero_days
        })

        # Train/Validation split logging
        wandb.log({
            "train_row_count": len(train_df),
            "val_row_count": len(val_df)
        })

        # --- Log Processed Artifact ---
        processed_data_artifact = wandb.Artifact(
            "processed-sales-data",
            type="processed_dataset",
            description="Daily aggregated sales data, split into train and validation sets."
        )
        processed_data_artifact.add_file("train.csv")
        processed_data_artifact.add_file("validation.csv")

        run.log_artifact(processed_data_artifact)
        print("Processed data artifact logged.")

preprocess_data()

wandb:   1 of 1 files downloaded.  


Total raw rows: 541909
Rows dropped due to missing CustomerID: 135080
Rows dropped due to non-positive Quantity: 8905
Rows dropped due to non-positive UnitPrice: 40
Remaining rows after cleaning: 397884

Resampled daily sales from 2010-12-01 to 2011-12-09
Total days after resampling: 374
Days with non-zero sales: 305

Train set rows: 314
Validation set rows: 60
Processed data artifact logged.


cleaned_row_count,▁
original_row_count,▁
resample_non_zero_days,▁
resample_total_days,▁
rows_dropped_missing_customer,▁
rows_dropped_negative_quantity,▁
rows_dropped_nonpositive_price,▁
train_row_count,▁
val_row_count,▁
cleaned_row_count,397884
original_row_count,541909


### TRAINING SCRIPT

In [5]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import wandb
import random
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# --- Dataset for univariate time series ---
class SalesDataset(torch.utils.data.Dataset):
    def __init__(self, data, sequence_length):
        self.data = torch.FloatTensor(data).view(-1)
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.data) - self.sequence_length - 1

    def __getitem__(self, index):
        return (self.data[index:index + self.sequence_length],
                self.data[index + self.sequence_length])


# --- LSTM Model Definition ---
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_layer_size=100, output_size=1):
        super().__init__()
        self.hidden_layer_size = hidden_layer_size
        self.lstm = nn.LSTM(input_size, hidden_layer_size)
        self.linear = nn.Linear(hidden_layer_size, output_size)
        self.hidden_cell = (torch.zeros(1, 1, self.hidden_layer_size),
                            torch.zeros(1, 1, self.hidden_layer_size))

    def forward(self, input_seq):
        lstm_out, self.hidden_cell = self.lstm(input_seq.permute(1, 0, 2), self.hidden_cell)
        predictions = self.linear(lstm_out[-1])
        return predictions

def set_seed(seed):
    # Python
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    # NumPy
    np.random.seed(seed)
    # Torch
    torch.manual_seed(seed)

In [7]:
# --- Main Training Pipeline ---
def train_forecaster():
    #hyperparameter config
    config = {
        "learning_rate": 0.0005,
        "epochs": 15,
        "sequence_length": 30,
        "hidden_layer_size": 50,
        "seed": 1
    }

    set_seed(config["seed"])

    #initialize run
    with wandb.init(project="sales-forecasting", job_type="training", config=config) as run:
        config = wandb.config

        # --- Load Preprocessed Data from W&B Artifact ---
        artifact = run.use_artifact('processed-sales-data:latest')
        artifact_dir = artifact.download()
        train_df = pd.read_csv(f"{artifact_dir}/train.csv")
        val_df = pd.read_csv(f"{artifact_dir}/validation.csv")

        # --- Extract and clean ---
        sales_data = pd.to_numeric(train_df["TotalSales"], errors="coerce").dropna().values
        val_data = pd.to_numeric(val_df["TotalSales"], errors="coerce").dropna().values
        print(f"Train: {len(sales_data)} rows | Val: {len(val_data)} rows")

        # --- Normalize ---
        scaler = StandardScaler()
        train_scaled = scaler.fit_transform(sales_data.reshape(-1, 1))
        val_scaled = scaler.transform(val_data.reshape(-1, 1))

        # --- Datasets & Loaders ---
        train_dataset = SalesDataset(train_scaled, config.sequence_length)
        val_dataset = SalesDataset(val_scaled, config.sequence_length)

        train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=False)
        val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=1, shuffle=False)

        # --- Model, Loss, Optimizer ---
        model = LSTMModel(hidden_layer_size=config.hidden_layer_size)
        loss_fn = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
        wandb.watch(model, log_freq=100)

        # --- Training Loop ---
        best_loss = float("inf")
        for epoch in range(config.epochs):
            model.train()
            train_losses = []
            for seq, label in train_loader:
                optimizer.zero_grad()
                model.hidden_cell = (
                    torch.zeros(1, 1, model.hidden_layer_size),
                    torch.zeros(1, 1, model.hidden_layer_size)
                )
                pred = model(seq.unsqueeze(-1))
                loss = loss_fn(pred, label.view(1, 1))
                loss.backward()
                optimizer.step()
                train_losses.append(loss.item())

            avg_train_loss = np.mean(train_losses)

            # --- Validation ---
            model.eval()
            val_losses = []
            with torch.no_grad():
                for seq, label in val_loader:
                    model.hidden_cell = (
                        torch.zeros(1, 1, model.hidden_layer_size),
                        torch.zeros(1, 1, model.hidden_layer_size)
                    )
                    pred = model(seq.unsqueeze(-1))
                    val_loss = loss_fn(pred, label)
                    val_losses.append(val_loss.item())

            avg_val_loss = np.mean(val_losses)

            # --- Log both losses ---
            wandb.log({
                "epoch": epoch,
                "train_loss": avg_train_loss,
                "val_loss": avg_val_loss
            })

            print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")

            # --- Save best model based on val_loss ---
            if avg_val_loss < best_loss:
                best_loss = avg_val_loss
                best_model_bundle = {
                    "model_state_dict": model.state_dict(),
                    "scaler": scaler,
                    "config": dict(config)
                }
                torch.save(best_model_bundle, "best_model_bundle.pth")
                artifact = wandb.Artifact(
                    "sales-forecasting-lstm",
                    type="model",
                    metadata={"epoch": epoch, "val_loss": best_loss}
                )
                artifact.add_file("best_model_bundle.pth")
                run.log_artifact(artifact, aliases=["best", f"epoch_{epoch}"])

                print(f" + Best model updated at epoch {epoch} (val_loss: {best_loss:.6f})")

        print("Training complete!")

train_forecaster()

wandb:   2 of 2 files downloaded.  


Train: 314 rows | Val: 60 rows


C:\Users\jagan\Documents\coding\MLops-DDOS\timeseries-forecasting-wandb\.venv\lib\site-packages\torch\nn\modules\loss.py:634: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch 0 | Train Loss: 0.975709 | Val Loss: 2.658241
 + Best model updated at epoch 0 (val_loss: 2.658241)
Epoch 1 | Train Loss: 0.952265 | Val Loss: 2.400313
 + Best model updated at epoch 1 (val_loss: 2.400313)
Epoch 2 | Train Loss: 0.916585 | Val Loss: 1.978424
 + Best model updated at epoch 2 (val_loss: 1.978424)
Epoch 3 | Train Loss: 0.807802 | Val Loss: 0.892111
 + Best model updated at epoch 3 (val_loss: 0.892111)
Epoch 4 | Train Loss: 0.659333 | Val Loss: 0.885048
 + Best model updated at epoch 4 (val_loss: 0.885048)
Epoch 5 | Train Loss: 0.606113 | Val Loss: 0.926603
Epoch 6 | Train Loss: 0.575579 | Val Loss: 0.945300
Epoch 7 | Train Loss: 0.553079 | Val Loss: 0.889166
Epoch 8 | Train Loss: 0.539047 | Val Loss: 0.829722
 + Best model updated at epoch 8 (val_loss: 0.829722)
Epoch 9 | Train Loss: 0.546796 | Val Loss: 0.733289
 + Best model updated at epoch 9 (val_loss: 0.733289)
Epoch 10 | Train Loss: 0.536135 | Val Loss: 0.633386
 + Best model updated at epoch 10 (val_loss: 0.63

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,██▇▆▃▃▂▂▂▂▂▁▁▁▁
val_loss,█▇▆▂▂▂▂▂▂▂▁▁▁▁▁
epoch,14
train_loss,0.48358
val_loss,0.56488


### TO BE RUN AFTER REGISTERING BEST MODEL IN REGISTRY

In [13]:
from sklearn.metrics import mean_squared_error
import wandb
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def forecast_from_best_model():
    run = wandb.init(project="sales-forecasting", job_type="inference")

    # Load best model artifact
    artifact = run.use_artifact("wandb-registry-wandb-registry-time-series-lstm/lstm-collection:best") # registry name: Time-series-lstm; collection name: lstm-collection
    artifact_dir = artifact.download()
    bundle = torch.load(f"{artifact_dir}/best_model_bundle.pth", map_location="cpu", weights_only=False)

    state_dict = bundle["model_state_dict"]
    scaler = bundle["scaler"]
    config = bundle["config"]

    # Load data
    data_artifact = run.use_artifact("processed-sales-data:latest")
    data_dir = data_artifact.download()
    train_df = pd.read_csv(f"{data_dir}/train.csv")
    val_df = pd.read_csv(f"{data_dir}/validation.csv")

    model = LSTMModel(hidden_layer_size=config["hidden_layer_size"])
    model.load_state_dict(state_dict)
    model.eval()

    for df, split in [(train_df, "train"), (val_df, "val")]:
        sales_data = pd.to_numeric(df["TotalSales"], errors="coerce").dropna().values
        sales_scaled = scaler.transform(sales_data.reshape(-1, 1))
        dataset = SalesDataset(sales_scaled, config["sequence_length"])

        preds = []
        for i in range(len(dataset)):
            with torch.no_grad():
                seq, _ = dataset[i]
                seq = seq.unsqueeze(0).unsqueeze(-1)
                model.hidden_cell = (
                    torch.zeros(1, 1, model.hidden_layer_size),
                    torch.zeros(1, 1, model.hidden_layer_size)
                )
                preds.append(model(seq).item())

        forecast = scaler.inverse_transform(np.array(preds).reshape(-1, 1)).flatten()
        actual = sales_data[config["sequence_length"]:config["sequence_length"] + len(forecast)]
        mse = mean_squared_error(actual, forecast)

        x = range(config["sequence_length"], config["sequence_length"] + len(forecast))
        plt.figure(figsize=(12, 6))
        plt.plot(x, actual, label="Actual Sales")
        plt.plot(x, forecast, label="Forecasted Sales")
        plt.title(f"Forecast vs Actual ({split}) | MSE: {mse:.4f}")
        plt.xlabel("Day Index")
        plt.ylabel("Sales")
        plt.legend()
        plt.grid(True, alpha=0.3)

        wandb.log({
            f"forecast_vs_actual_plot_{split}": wandb.Image(plt),
            f"forecast_mse_{split}": mse
        })
        plt.close()

    print("Forecasts and MSEs logged for both train and validation sets.")

forecast_from_best_model()

wandb:   1 of 1 files downloaded.  
wandb:   2 of 2 files downloaded.  


Forecasts and MSEs logged for both train and validation sets.
